# Laboratório Pratico - Agentes Inteligentes
---
Nome: Davi Horácio de Souza

Matrícula: 20250059934

## Ajustando ambiente

In [1]:
import os
import sys
import random
import collections

## Motivação


O estudo de **Agentes Inteligentes** é a base da Inteligência Artificial moderna, que pode ser compreendida essencialmente como a ciência do design de agentes. Estudar esses sistemas nos ensina a projetar softwares ou robôs capazes de perceber seus ambientes através de sensores e agir de forma autônoma por meio de atuadores. Ao focar na construção de um **agente racional** — aquele que sempre busca tomar a ação que maximiza a sua medida de desempenho esperada — aprendemos um conjunto de princípios de design universais aplicáveis a qualquer ambiente imaginável. Desde carros autônomos navegando em rodovias até sistemas virtuais de diagnóstico médico dominar a estrutura dos agentes nos permite desenvolver soluções robustas que não apenas reagem aos dados, mas que planejam, otimizam recursos e aprendem continuamente para se adaptar a problemas complexos e incertos no mundo real.


No contexto deste notebook, o problema do aspirador é valioso porque ele reduz a complexidade do tema a uma estrutura pequena o bastante para estudar três perguntas fundamentais:

1. O que o agente percebe?
2. Como ele escolhe uma ação?
3. Como o ambiente reage a essa ação?

## Objetivos de Aprendizagem


* Revisar conceitos de orientação a objeto em Python.

1.   Item da lista
2.   Item da lista


* Entender o papel de Thing, Agent e Environment.
* Descrever o funcionamento de TrivialVacuumEnvironment e sua medida de desempenho.
* Diferenciar agentes tabelados, reflexos e baseados em modelo.
* Comparar como memória e representação afetam a eficiência do agente.

## Fundamentação Teórica

### Definição formal de agente


Um agente pode ser compreendido como qualquer entidade que percebe o seu ambiente por meio de sensores e age sobre esse ambiente por meio de atuadores. Matematicamente, o comportamento de um agente é descrito pela **função do agente**, que mapeia sequências de percepções para ações:

$$
\text{agent}: P^* \rightarrow A
$$

em que $P^*$ representa o conjunto de todas as sequências possíveis de percepções (ou seja, o histórico completo de tudo o que o agente já percebeu até o momento) e $A$ representa o conjunto de ações possíveis. É fundamental distinguir a *função do agente*, que é uma descrição matemática abstrata, do **programa do agente**, que é a implementação concreta dessa função rodando dentro de um sistema físico ou computacional (a arquitetura).

### Medida de desempenho

Em vez de perguntar apenas se o agente "funciona", a Inteligência Artificial adota uma abordagem baseada no consequencialismo: nós avaliamos o comportamento de um agente pelas consequências de suas ações. Quando um agente atua, ele faz com que o ambiente passe por uma sequência de estados; se essa sequência for desejável, o agente teve um bom desempenho.

A partir dessa medida, definimos um **agente racional** como aquele que, para cada sequência de percepções, escolhe a ação que tem a expectativa de maximizar a sua medida de desempenho. Como regra geral de design, a medida de desempenho deve ser definida de acordo com o que você *realmente deseja que seja alcançado no ambiente*, e não de acordo com a forma como você acha que o agente deveria se comportar.

## Implementação Prática

Neste notebook, a implementação do conceito teórico de que um **agente = arquitetura + programa**. A partir desse momento iremos organizar a implementação em:

- a estrutura básica do agente;
- a estrutura básica do ambiente;
- agentes concretos para o mundo do aspirador;
- ambientes concretos que definem percepção, ação e custo.

[Material suplementar de OOP em Python](https://www.geeksforgeeks.org/python/python-oops-concepts/)

In [2]:
class Thing:
    """This represents any physical object that can appear in an Environment.
    You subclass Thing to get the things you want. Each thing can have a
    .__name__  slot (used for output only)."""

    def __repr__(self):
        return '<{}>'.format(getattr(self, '__name__', self.__class__.__name__))

    def is_alive(self):
        """Things that are 'alive' should return true."""
        return hasattr(self, 'alive') and self.alive

    def show_state(self):
        """Display the agent's internal state. Subclasses should override."""
        print("I don't know how to show_state.")

    def display(self, canvas, x, y, width, height):
        """Display an image of this Thing on the canvas."""
        pass

A classe Thing é importante porque ela define a abstração mais básica dos objetos que podem existir em um ambiente, servindo como ponto comum para agentes, sujeira, paredes e outros elementos do mundo. Com isso, o projeto consegue tratar diferentes entidades de forma uniforme, reaproveitando atributos e comportamentos gerais, enquanto subclasses especializam o que cada tipo de objeto realmente faz.

### Classe `Agent`

A implementação de `Agent` concentra a ideia mínima de um agente no projeto:

In [3]:
class Agent(Thing):
    def __init__(self, program=None):
        self.alive = True
        self.bump = False
        self.holding = []
        self.performance = 0
        # Use callable() for a more robust check
        if program is None or not callable(program):
            def program(percept):
                # This is the default interactive program if no valid program is provided
                return eval(input('Percept={}; action? '.format(percept)))
        self.program = program

    def can_grab(self, thing):
        return False

Pontos centrais:

- `program` recebe uma percepção e devolve uma ação.
- `performance` acumula a medida de desempenho.
- `can_grab` pode ser sobrescrito por subclasses mais especializadas.


<details>
 <summary>Explicação</summary>

* `class Agent(Thing):`
Define a classe `Agent` como subclasse de `Thing`. Ou seja, todo agente também é uma “coisa” que pode existir no ambiente.
* `def __init__(self, program=None):`
Construtor da classe. Recebe opcionalmente a função que implementa o comportamento do agente.
* `self.alive = True`
Marca o agente como vivo/ativo no ambiente.
* `self.bump = False`
Inicializa o estado de colisão. Esse atributo costuma indicar se o agente bateu em um obstáculo.
* `self.holding = []`
Cria a lista de objetos que o agente está carregando.
* `self.performance = 0`
Inicializa a medida de desempenho do agente.
* `if program is None or not callable(program):`
Verifica se o `program` foi fornecido e se ele realmente pode ser chamado como função.
* `self.program = program`
Guarda a função de decisão no objeto. Esse é o atributo mais importante da classe.
* `def can_grab(self, thing):`
Declara um método que pergunta se o agente consegue pegar um objeto específico.
* **Docstring do método:**
Explica que subclasses podem sobrescrever esse comportamento.
* `return False`
Valor padrão: por padrão, um `Agent` não pega objetos. Subclasses especializadas podem mudar isso.
</details>

### Classe `Environment`

A classe Environment pode ser entendida como o “mundo” onde os agentes vivem e agem: ela guarda os objetos presentes, informa ao agente o que ele consegue perceber e aplica as consequências de cada ação. Em vez de o agente alterar o mundo diretamente, o ambiente é quem controla as regras da simulação, coordenando o ciclo de perceber, agir e atualizar o estado do sistema.

In [4]:
class Environment:

    def __init__(self):
        self.things = []
        self.agents = []

    def percept(self, agent):
        raise NotImplementedError

    def execute_action(self, agent, action):
        raise NotImplementedError

    def default_location(self, thing):
        return None

    def exogenous_change(self):
        pass

    def is_done(self):
        return not any(agent.is_alive() for agent in self.agents)

    def step(self):
        if not self.is_done():
            actions = []
            for agent in self.agents:
                if agent.alive:
                    actions.append(agent.program(self.percept(agent)))
                else:
                    actions.append("")
            for (agent, action) in zip(self.agents, actions):
                self.execute_action(agent, action)
            self.exogenous_change()

    def run(self, steps=1000):
        for step in range(steps):
            if self.is_done():
                return
            self.step()

    def add_thing(self, thing, location=None):
        if not isinstance(thing, Thing):
            thing = Agent(thing)
        if thing in self.things:
            print("Can't add the same thing twice")
        else:
            thing.location = location if location is not None else self.default_location(thing)
            self.things.append(thing)
            if isinstance(thing, Agent):
                thing.performance = 0
                self.agents.append(thing)

<details>
 <summary>
 Por que o método execute_action foi implementado no ambiente?
 </summary>

  A implementação do método `execute_action` na classe `Environment` (Ambiente) ocorre porque, de acordo com a fundamentação teórica, **é o ambiente que é afetado pelas ações do agente e sofre mudanças de estado**.

  Na estrutura dos sistemas de Inteligência Artificial, a interação funciona da seguinte forma:
  * O agente percebe o ambiente por meio de sensores, toma uma decisão através de seu programa e age sobre o ambiente utilizando os seus **atuadores**.
  * Quando o agente executa essa ação (como mover-se, enviar um pacote de rede ou aspirar uma sujeira), essa ação faz com que o ambiente passe por uma nova sequência de estados.

  Portanto, no design do código, ocorre uma separação clara de responsabilidades: o agente toma a decisão e diz *o que* quer fazer, mas é o ambiente que deve conter a função `execute_action` para receber essa decisão, processar a ação através dos atuadores virtuais e aplicar as alterações correspondentes nas regras e no estado do próprio mundo simulado.

 </details>




<details>
 <summary>
 Considerando as propriedades dos ambientes de tarefas. Qual a função do exogenous_change?
 </summary>

  A função `exogenous_change` tem o papel de introduzir alterações no estado do sistema que acontecem de forma independente da ação imediata do agente. De acordo com as propriedades teóricas dos ambientes de tarefas, essa chamada no código é utilizada para modelar as seguintes características:

  *   **Ambientes Dinâmicos (Dynamic)**: A função permite simular que o ambiente pode sofrer alterações por conta própria com a passagem do tempo, inclusive enquanto o agente ainda está deliberando sobre qual ação tomar.
  *   **Ambientes Não-determinísticos (Nondeterministic)**: A função assegura que o próximo estado do ambiente não seja completamente e exclusivamente definido pelo estado atual somado à ação executada pelo agente. Isso possibilita a inclusão de eventos externos, complexos e incertos na simulação, como o surgimento aleatório de uma nova sujeira no ambiente do aspirador de pó ou um pneu que estoura inesperadamente no ambiente de um táxi.

</details>


### Classes do Mundo do Aspirador

Este é o ambiente usado em todo o tutorial. Ele possui apenas duas localizações e devolve ao agente uma percepção na forma `(posição, estado_do_local)`.

In [5]:
class TrivialVacuumEnvironment(Environment):
    def __init__(self):
        super().__init__()
        self.status = {loc_A: random.choice(['Clean', 'Dirty']),
                       loc_B: random.choice(['Clean', 'Dirty'])}

    def percept(self, agent):
        return agent.location, self.status[agent.location]

    def execute_action(self, agent, action):
        if action == 'Right':
            agent.location = loc_B
            agent.performance -= 1
        elif action == 'Left':
            agent.location = loc_A
            agent.performance -= 1
        elif action == 'Suck':
            if self.status[agent.location] == 'Dirty':
                agent.performance += 10
            self.status[agent.location] = 'Clean'

TrivialVacuumEnvironment é o ambiente mais simples do problema do aspirador. Ele possui apenas dois locais, loc_A e loc_B, e em cada um deles pode haver sujeira ou não. Sua função no tutorial é mostrar, de forma mínima, como um ambiente entrega percepções ao agente e como atualiza o mundo quando o agente executa ações como Left, Right e Suck. Além disso, ele já define uma medida de desempenho clara: mover para a esquerda ou direita custa -1 ponto, enquanto limpar um local sujo com Suck rende +10 pontos. Isso torna o ambiente ideal para comparar agentes diferentes de forma simples e objetiva.

### Função `TraceAgent`

In [6]:
def TraceAgent(agent):
    old_program = agent.program

    def new_program(percept):
        action = old_program(percept)
        print('{} perceives {} and does {}'.format(agent, percept, action))
        return action

    agent.program = new_program
    return agent

TraceAgent é um empacotador de depuração. Basicamente, ele modifica o program de um agente para imprimir, a cada passo, qual percepção o agente recebeu e qual ação escolheu. Isso é útil para acompanhar o raciocínio do agente durante a execução sem alterar a lógica principal dele.


## Exemplo prático

### Agente baseado em tabela

O programa `TableDrivenAgentProgram` usa a sequência completa de percepções como chave de consulta em uma tabela. Isso é conceitualmente simples, mas pouco escalável, porque a tabela precisa prever muitos históricos possíveis.

In [7]:
def TableDrivenAgentProgram(table):
    percepts = []

    def program(percept):
        percepts.append(percept)
        action = table.get(tuple(percepts))
        return action

    return program

In [8]:
loc_A, loc_B = 'A', 'B'

example_table = {
    ((loc_A, 'Dirty'),): 'Suck',
    ((loc_A, 'Dirty'), (loc_A, 'Clean')): 'Right',
    ((loc_A, 'Dirty'), (loc_A, 'Clean'), (loc_B, 'Dirty')): 'Suck'
}

table_program = TableDrivenAgentProgram(example_table)

print(table_program((loc_A, 'Dirty')))
print(table_program((loc_A, 'Clean')))
print(table_program((loc_B, 'Dirty')))

Suck
Right
Suck


In [9]:
def TableDrivenVacuumAgent():
    table = {((loc_A, 'Clean'),): 'Right',
             ((loc_A, 'Dirty'),): 'Suck',
             ((loc_B, 'Clean'),): 'Left',
             ((loc_B, 'Dirty'),): 'Suck',
             ((loc_A, 'Dirty'), (loc_A, 'Clean')): 'Right',
             ((loc_A, 'Clean'), (loc_B, 'Dirty')): 'Suck',
             ((loc_B, 'Clean'), (loc_A, 'Dirty')): 'Suck',
             ((loc_B, 'Dirty'), (loc_B, 'Clean')): 'Left',
             ((loc_A, 'Dirty'), (loc_A, 'Clean'), (loc_B, 'Dirty')): 'Suck',
             ((loc_B, 'Dirty'), (loc_B, 'Clean'), (loc_A, 'Dirty')): 'Suck'
    }
    return Agent(TableDrivenAgentProgram(table))

In [10]:
table_environment = TrivialVacuumEnvironment()
table_environment.status = {loc_A: 'Dirty', loc_B: 'Dirty'}
table_agent = TraceAgent(TableDrivenVacuumAgent())
table_environment.add_thing(table_agent, loc_A)

for step_num in range(5):
    print(f'Passo {step_num + 1}')
    table_environment.step()
    print('Estado:', table_environment.status, '| posição:', table_agent.location, '| desempenho:', table_agent.performance)
    print('\n')

table_result = (dict(table_environment.status), table_agent.location, table_agent.performance)

Passo 1
<Agent> perceives ('A', 'Dirty') and does Suck
Estado: {'A': 'Clean', 'B': 'Dirty'} | posição: A | desempenho: 10


Passo 2
<Agent> perceives ('A', 'Clean') and does Right
Estado: {'A': 'Clean', 'B': 'Dirty'} | posição: B | desempenho: 9


Passo 3
<Agent> perceives ('B', 'Dirty') and does Suck
Estado: {'A': 'Clean', 'B': 'Clean'} | posição: B | desempenho: 19


Passo 4
<Agent> perceives ('B', 'Clean') and does None
Estado: {'A': 'Clean', 'B': 'Clean'} | posição: B | desempenho: 19


Passo 5
<Agent> perceives ('B', 'Clean') and does None
Estado: {'A': 'Clean', 'B': 'Clean'} | posição: B | desempenho: 19




<details>
<summary>Como poderíamos consertar o problema das ações "None"?</summary>

Para consertar o problema das ações None adicionando as combinações manualmente, é necessário expandir o dicionário table para incluir todas as sequências cronológicas de percepções possíveis que o agente pode acumular. O erro ocorre porque, à medida que o tempo passa, o agente gera históricos mais longos (como tuplas de 4 ou 5 percepções) que não possuem mapeamento na tabela, resultando em um retorno vazio pelo método .get(). A solução exige prever cada ramificação do ambiente e escrever chaves exaustivas para cada passo da simulação, garantindo que cada estágio da experiência do agente tenha uma instrução definida, como Right, Suck ou NoOp.

*  ((loc_A, 'Dirty'), (loc_A, 'Clean'), (loc_B, 'Dirty'), (loc_B, 'Clean')): 'NoOp',

* ((loc_A, 'Dirty'), (loc_A, 'Clean'), (loc_B, 'Dirty'), (loc_B, 'Clean'), (loc_B, 'Clean')): 'NoOp'

</details>

<details>
<summary>
  Por que agentes baseados em tabelas são estruturalmente impraticáveis?
</summary>

  Os agentes baseados em tabelas são estruturalmente impraticáveis porque a quantidade de entradas necessárias para mapear todo o histórico possível de percepções para ações sofre de uma explosão combinatória exponencial. Matematicamente, à medida que o tempo de vida do agente avança, o tamanho da tabela cresce de forma tão dramática que, mesmo em ambientes delimitados como o xadrez — que exigiria pelo menos $10^{150}$ entradas, superando o número de átomos no universo observável — ou na condução de um táxi autônomo, que geraria mais de $10^{600.000.000.000}$ entradas em apenas uma hora de processamento visual contínuo, o volume de dados exigido se torna astronômico. Devido a essa imensa magnitude, a abordagem de design baseada em tabelas está fadada ao fracasso por três motivos absolutos: nenhum sistema físico no universo possui espaço de armazenamento suficiente para abrigar tal estrutura, nenhum projetista humano teria tempo hábil para criá-la e preenchê-la manualmente, e nenhum agente conseguiria viver ou interagir por tempo suficiente para aprender todas essas entradas corretas exclusivamente por meio de sua própria experiência.
</details>

### Agente reativo simples

O `ReflexVacuumAgent` abandona a tabela de históricos e reage apenas ao percepto atual. Isso reduz a complexidade da representação, desde que o percepto já contenha informação suficiente para decidir.

In [11]:
def ReflexVacuumAgent():
    def program(percept):
        location, status = percept
        if status == 'Dirty':
            return 'Suck'
        elif location == loc_A:
            return 'Right'
        elif location == loc_B:
            return 'Left'

    return Agent(program)

In [12]:
reflex_environment = TrivialVacuumEnvironment()
reflex_environment.status = {loc_A: 'Dirty', loc_B: 'Dirty'}
reflex_agent = TraceAgent(ReflexVacuumAgent())
reflex_environment.add_thing(reflex_agent, loc_A)

for passo in range(5):
    print(f'Passo {passo + 1}')
    reflex_environment.step()
    print('Estado técnico:', reflex_environment.status, '| posição:', reflex_agent.location, '| desempenho:', reflex_agent.performance)
    print('\n')

resultado_reflexo = (dict(reflex_environment.status), reflex_agent.location, reflex_agent.performance)

Passo 1
<Agent> perceives ('A', 'Dirty') and does Suck
Estado técnico: {'A': 'Clean', 'B': 'Dirty'} | posição: A | desempenho: 10


Passo 2
<Agent> perceives ('A', 'Clean') and does Right
Estado técnico: {'A': 'Clean', 'B': 'Dirty'} | posição: B | desempenho: 9


Passo 3
<Agent> perceives ('B', 'Dirty') and does Suck
Estado técnico: {'A': 'Clean', 'B': 'Clean'} | posição: B | desempenho: 19


Passo 4
<Agent> perceives ('B', 'Clean') and does Left
Estado técnico: {'A': 'Clean', 'B': 'Clean'} | posição: A | desempenho: 18


Passo 5
<Agent> perceives ('A', 'Clean') and does Right
Estado técnico: {'A': 'Clean', 'B': 'Clean'} | posição: B | desempenho: 17




### Agente baseado em modelo

O `ModelBasedVacuumAgent` mantém um pequeno estado interno com o que já sabe sobre `loc_A` e `loc_B`. Assim, ele consegue parar com `NoOp` quando conclui que todo o ambiente está limpo.

In [13]:
def ModelBasedVacuumAgent():
    model = {loc_A: None, loc_B: None}

    def program(percept):
        location, status = percept
        model[location] = status
        if model[loc_A] == model[loc_B] == 'Clean':
            return 'NoOp'
        elif status == 'Dirty':
            return 'Suck'
        elif location == loc_A:
            return 'Right'
        elif location == loc_B:
            return 'Left'

    return Agent(program)

In [14]:
model_environment = TrivialVacuumEnvironment()
model_environment.status = {loc_A: 'Dirty', loc_B: 'Dirty'}
model_agent = TraceAgent(ModelBasedVacuumAgent())
model_environment.add_thing(model_agent, loc_A)

for passo in range(5):
    print(f'Passo {passo + 1}')
    model_environment.step()
    print('Estado técnico:', model_environment.status, '| posição:', model_agent.location, '| desempenho:', model_agent.performance)
    print('\n')

resultado_modelo_demo = (dict(model_environment.status), model_agent.location, model_agent.performance)

Passo 1
<Agent> perceives ('A', 'Dirty') and does Suck
Estado técnico: {'A': 'Clean', 'B': 'Dirty'} | posição: A | desempenho: 10


Passo 2
<Agent> perceives ('A', 'Clean') and does Right
Estado técnico: {'A': 'Clean', 'B': 'Dirty'} | posição: B | desempenho: 9


Passo 3
<Agent> perceives ('B', 'Dirty') and does Suck
Estado técnico: {'A': 'Clean', 'B': 'Clean'} | posição: B | desempenho: 19


Passo 4
<Agent> perceives ('B', 'Clean') and does NoOp
Estado técnico: {'A': 'Clean', 'B': 'Clean'} | posição: B | desempenho: 19


Passo 5
<Agent> perceives ('B', 'Clean') and does NoOp
Estado técnico: {'A': 'Clean', 'B': 'Clean'} | posição: B | desempenho: 19




### Comparação entre os Agentes

In [15]:
def execute_agent(factory, initial_state, steps=5):
    environment = TrivialVacuumEnvironment()
    environment.status = dict(initial_state)
    agent = factory()
    environment.add_thing(agent, loc_A)
    environment.run(steps)
    return environment.status, agent.location, agent.performance

scenario = {loc_A: 'Dirty', loc_B: 'Dirty'}

table_result = execute_agent(TableDrivenVacuumAgent, scenario)
reflex_result = execute_agent(ReflexVacuumAgent, scenario)
model_result = execute_agent(ModelBasedVacuumAgent, scenario)

print('ReflexVacuumAgent:', reflex_result)
print('ModelBasedVacuumAgent:', model_result)

ReflexVacuumAgent: ({'A': 'Clean', 'B': 'Clean'}, 'B', 17)
ModelBasedVacuumAgent: ({'A': 'Clean', 'B': 'Clean'}, 'B', 19)


<details>
<summary>Qual a distinção entre Agentes reativos simples e reativos baseados em modelo?
</summary>

A principal distinção entre os dois tipos de agentes reside na forma como lidam com a observabilidade do ambiente e com o histórico de percepções. Os **agentes reativos simples** selecionam suas ações baseando-se exclusivamente na percepção atual por meio de regras diretas de condição-ação, ignorando completamente o histórico de tudo o que já perceberam, o que limita o seu funcionamento adequado apenas a ambientes que são totalmente observáveis. Em contrapartida, os **agentes reativos baseados em modelo** são projetados para lidar com ambientes parcialmente observáveis, mantendo um estado interno que utiliza o histórico de percepções para rastrear os aspectos do mundo que não podem ser vistos no momento. Para que esse estado interno seja atualizado continuamente, esses agentes incorporam duas formas de conhecimento: um modelo de transição, que descreve como o mundo evolui e como as ações do agente o afetam, e um modelo de sensor, que detalha como o estado real do mundo se reflete naquilo que os sensores conseguem captar.</details>

<details>
<summary>Por que o ReflexVacuumAgent obteve um desempenho menor (17) que o ModelBasedVacuumAgent (19) no cenário testado?
</summary>

A diferença de desempenho reside na capacidade de representação do estado do mundo: enquanto o ReflexVacuumAgent toma decisões baseadas exclusivamente no percepto imediato, o ModelBasedVacuumAgent mantém uma memória interna (modelo) do que já foi explorado e limpo. No cenário testado, ambos atingem a pontuação máxima de 19 pontos no terceiro passo, após limparem as duas localizações ; contudo, por não possuir memória, o agente reflexo não "sabe" que o ambiente já está totalmente limpo e continua se movendo entre as salas nos passos seguintes, sofrendo penalidades de -1 ponto por movimento desnecessário, o que reduz seu placar final para 17. Já o agente baseado em modelo atualiza seu estado interno e, ao concluir que não há mais sujeira, executa a ação NoOp (nenhuma operação), que não possui custo de movimento, preservando assim seu desempenho máximo de 19 pontos até o final da simulação.
</details>

## Desafio


O objetivo deste desafio é expandir o cenário clássico de duas salas para um ambiente linear de três localizações (**A**, **B** e **C**), elevando a complexidade da tomada de decisão e testando a escalabilidade das arquiteturas de agentes. Enquanto no mundo trivial a percepção imediata costuma ser suficiente, em um arranjo de três salas, o agente posicionado no centro (sala **B**) enfrenta uma ambiguidade espacial: sem memória, ele não consegue determinar se a sujeira remanescente está à sua esquerda ou à sua direita.

Para solucionar o problema, siga estas diretrizes sucintas:

* **Expansão do Ambiente:** Produza uma classe `TriVacuumEnvironment` para incluir `loc_C` no dicionário de estados e ajuste o método `execute_action` para gerenciar os novos limites de movimentação linear entre A-B e B-C.

* **Otimização do Agente Reflexo:** Para o `ReflexTriVacuumAgent`, que carece de memória, implemente uma estratégia de varredura cíclica ou escolha aleatória em 'B' para evitar que o robô fique estagnado no centro do ambiente.

* **Ajuste do Modelo Interno:** No `ModelBasedTriVacuumAgent`, expanda o dicionário `model` para rastrear as três salas e atualize a condição de parada `NoOp` para verificar se todo o vetor de estados consta como limpo.

* **Lógica de Navegação em B:** Implemente uma regra de decisão na sala central que consulte o modelo interno: o agente deve se mover para a direção onde o estado da sala ainda seja "Dirty" ou desconhecido (`None`).



#### Criando o ambiente adicionando o loc_C

In [16]:
loc_C = 'C'

class TriVacuumEnvironment(Environment):
  def __init__(self):
      super().__init__()
      self.status = {loc_A: random.choice(['Dirty', 'Clean']),
                    loc_B: random.choice(['Dirty', 'Clean']),
                    loc_C: random.choice(['Dirty', 'Clean'])}

  def percept(self, agent):
          return (agent.location, self.status[agent.location])

  def execute_action(self, agent, action):
    if action == 'Right' and agent.location == loc_A:
      agent.location = loc_B
      agent.performance -= 1

    elif action == 'Right' and agent.location == loc_B:
      agent.location = loc_C
      agent.performance -= 1

    elif action == 'Left' and agent.location == loc_C:
      agent.location = loc_B
      agent.performance -= 1

    elif action == 'Left' and agent.location == loc_B:
      agent.location = loc_A
      agent.performance -= 1

    elif action == "Suck" and self.status[agent.location] == 'Dirty':
      agent.performance += 10
      self.status[agent.location] = 'Clean'




#### Criando o agente reflexivo simples

In [17]:
def ReflexTriVacuumAgent():
    def program(percept):
        location, status = percept
        if status == 'Dirty':
          return 'Suck'
        elif location == loc_A:
          return 'Right'
        elif location == loc_B:
          return random.choice(['Left', 'Right'])
        elif location == loc_C:
          return 'Left'

    return Agent(program)

#### Aplicando o agente

In [18]:
reflextri_environment = TriVacuumEnvironment()
reflextri_environment.status = {loc_A: 'Dirty', loc_B: 'Dirty', loc_C: 'Dirty'}
reflextri_agent = TraceAgent(ReflexTriVacuumAgent())
reflextri_environment.add_thing(reflextri_agent, loc_A)

for passo in range(10):
    print(f'Passo {passo + 1}')
    reflextri_environment.step()
    print('Estado técnico:', reflextri_environment.status, '| posição:', reflextri_agent.location, '| desempenho:', reflextri_agent.performance)
    print('\n')

resultado_reflexotri = (dict(reflextri_environment.status), reflextri_agent.location, reflextri_agent.performance)

Passo 1
<Agent> perceives ('A', 'Dirty') and does Suck
Estado técnico: {'A': 'Clean', 'B': 'Dirty', 'C': 'Dirty'} | posição: A | desempenho: 10


Passo 2
<Agent> perceives ('A', 'Clean') and does Right
Estado técnico: {'A': 'Clean', 'B': 'Dirty', 'C': 'Dirty'} | posição: B | desempenho: 9


Passo 3
<Agent> perceives ('B', 'Dirty') and does Suck
Estado técnico: {'A': 'Clean', 'B': 'Clean', 'C': 'Dirty'} | posição: B | desempenho: 19


Passo 4
<Agent> perceives ('B', 'Clean') and does Left
Estado técnico: {'A': 'Clean', 'B': 'Clean', 'C': 'Dirty'} | posição: A | desempenho: 18


Passo 5
<Agent> perceives ('A', 'Clean') and does Right
Estado técnico: {'A': 'Clean', 'B': 'Clean', 'C': 'Dirty'} | posição: B | desempenho: 17


Passo 6
<Agent> perceives ('B', 'Clean') and does Left
Estado técnico: {'A': 'Clean', 'B': 'Clean', 'C': 'Dirty'} | posição: A | desempenho: 16


Passo 7
<Agent> perceives ('A', 'Clean') and does Right
Estado técnico: {'A': 'Clean', 'B': 'Clean', 'C': 'Dirty'} | posiçã

### Agente baseado em modelo

In [19]:
def ModelBasedVacuumTriAgent():
    model = {loc_A: None, loc_B: None, loc_C: None}

    def program(percept):
        location, status = percept
        model[location] = status
        if model[loc_A] == model[loc_B] == model[loc_C]== 'Clean':
            return 'NoOp'
        elif status == 'Dirty':
            return 'Suck'
        elif location == loc_A:
            return 'Right'
        elif location == loc_C:
            return 'Left'
        elif location == loc_B:
            if model[loc_A] != 'Clean':
                return 'Left'
            else:
                return 'Right'

    return Agent(program)

#### Executando o agente baseado em modelo

In [20]:
model_trienvironment = TriVacuumEnvironment()
model_trienvironment.status = {loc_A: 'Dirty', loc_B: 'Dirty', loc_C: 'Dirty'}
model_triagent = TraceAgent(ModelBasedVacuumTriAgent())
model_trienvironment.add_thing(model_triagent, loc_A)

for passo in range(10):
    print(f'Passo {passo + 1}')
    model_trienvironment.step()
    print('Estado técnico:', model_trienvironment.status, '| posição:', model_triagent.location, '| desempenho:', model_triagent.performance)
    print('\n')

resultado_modelotri_demo = (dict(model_trienvironment.status), model_triagent.location, model_triagent.performance)

Passo 1
<Agent> perceives ('A', 'Dirty') and does Suck
Estado técnico: {'A': 'Clean', 'B': 'Dirty', 'C': 'Dirty'} | posição: A | desempenho: 10


Passo 2
<Agent> perceives ('A', 'Clean') and does Right
Estado técnico: {'A': 'Clean', 'B': 'Dirty', 'C': 'Dirty'} | posição: B | desempenho: 9


Passo 3
<Agent> perceives ('B', 'Dirty') and does Suck
Estado técnico: {'A': 'Clean', 'B': 'Clean', 'C': 'Dirty'} | posição: B | desempenho: 19


Passo 4
<Agent> perceives ('B', 'Clean') and does Right
Estado técnico: {'A': 'Clean', 'B': 'Clean', 'C': 'Dirty'} | posição: C | desempenho: 18


Passo 5
<Agent> perceives ('C', 'Dirty') and does Suck
Estado técnico: {'A': 'Clean', 'B': 'Clean', 'C': 'Clean'} | posição: C | desempenho: 28


Passo 6
<Agent> perceives ('C', 'Clean') and does NoOp
Estado técnico: {'A': 'Clean', 'B': 'Clean', 'C': 'Clean'} | posição: C | desempenho: 28


Passo 7
<Agent> perceives ('C', 'Clean') and does NoOp
Estado técnico: {'A': 'Clean', 'B': 'Clean', 'C': 'Clean'} | posição

## Key Takeways

Ao final do tutorial, espera-se que o aluno compreenda como conceitos de orientação a objetos em Python são usados para estruturar agentes e ambientes, e consiga explicar a evolução de um agente tabelado para um agente reflexo e depois para um agente baseado em modelo. Também se espera que ele entenda como o ambiente define percepções, ações e desempenho, e por que o uso de memória interna pode tornar a tomada de decisão mais eficiente.

## Referências

1. Russell, S. & Norvig, P. (2010). Artificial Intelligence: A Modern Approach. Prentice Hall.
2. Wikipedia: Hill-climbing algorithm
3. GeeksforGeeks: N-Queen Problem using Hill Climbing